# M0 pilot: session runbook (D-033)

Runs the M0 pilot, the 244 BB codes at n ≤ 72 at `P_PILOT` with the pinned BP+OSD (D-031), as a chain of Kaggle sessions. All logic is in `qecscreen.evaluate.pilot`; these cells only set configuration and call it. The same runbook is in `spec/architecture.md §2`.

**Every session**
- Notebook settings: Internet **on** (cell 1 installs from GitHub), accelerator **none**.
- Cell 1: `QECSCREEN_SHA` is the full 40-hex commit you were given, and **the same commit every session**. A later session refuses to start if the commit, ldpc or stim differs from session 1's. There is no override.
- Run it with **Save Version → Save & Run All (Commit)**, not interactively, so the session runs to its end and `/kaggle/working` is kept as that version's output.

**Session 1: the probe**
1. `PREVIOUS = None`, `COST_GATE = ""`.
2. Save & Run All. It runs 12 codes at evenly spaced ranks of `n * d_upper`, the largest included, for 3 h, then stops. Its work counts toward the pilot.
3. Download the version's output folder `m0-pilot/`. Locally, from the repository root:
   `from qecscreen.evaluate.pilot import pilot_cost_report, format_cost_report`
   `print(format_cost_report(pilot_cost_report("<download>/m0-pilot", "evidence/calibration/2026-09-27-7e91f82")))`
4. Record the report in `spec/evals.md §7`. Continue only if you approve it.

**Session 2 onward**
1. In the editor: **Add Input → Your Work → Notebooks**, and add this notebook's **latest** version, and only that one: two attached pilot outputs are refused. Remove older versions from the inputs.
2. `PREVIOUS` is the directory the input is mounted at, as the Input panel shows it, under `/kaggle/input` (for example `"/kaggle/input/<this-notebook>"`). The runner finds the one `m0-pilot` below it, copies it into `/kaggle/working/m0-pilot`, and checks every file against what the previous session recorded.
3. `COST_GATE` is the title of the `spec/evals.md §7` entry that approved the probe's cost report. It is recorded in the manifest.
4. Save & Run All. No code starts after 10.5 h; the running ones stop after their current batch, and the session ends.
5. Read the printed summary: codes done, partial, not started, stale-restarted, died and failed, and decode core-hours this session and in all. A code that died is retried once, in the next session. A code that died twice is failed and needs your decision.

**When the summary shows 244 done**
Download the last output. Locally, from the repository root:
`from qecscreen.evaluate.pilot import assemble_measurements`
`assemble_measurements("<download>/m0-pilot", "data")`
writes `data/m0_measurements.parquet`. It refuses while any code is unfinished.

**Never** change `QECSCREEN_SHA` mid-pilot, edit anything inside `m0-pilot/`, or run a session interactively and close the tab.

In [ ]:
# 1 - Install qecscreen, pinned by commit SHA (D-028, D-030, spec/architecture.md §2).
# QECSCREEN_SHA is the full 40-hex commit to run, never a branch or a tag.
# Cell 2 checks that the installed code really is this commit (D-017).
#
# The repo is public (D-030), so no secret is needed. If a GITHUB_TOKEN secret
# is attached (Kaggle: Add-ons -> Secrets; Colab: the key icon), it is used;
# otherwise the install uses the plain https URL. A missing secret is not an
# error. Never paste a token into this notebook: its cells and outputs are kept
# in Kaggle's version history and in every copy of it.
import re
import subprocess
import sys

QECSCREEN_SHA = ""
assert re.fullmatch(r"[0-9a-f]{40}", QECSCREEN_SHA), "set QECSCREEN_SHA to a full 40-hex commit"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:  # not on Kaggle, or no such secret attached
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:  # not on Colab, or no such secret / no notebook access
        token = None

auth = f"{token}@" if token else ""
print("installing", "with the GITHUB_TOKEN secret" if token else "from the public URL, no token")
proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     f"git+https://{auth}github.com/Pushkar0997/qecscreen@{QECSCREEN_SHA}"],
    capture_output=True,
    text=True,
)
output = proc.stdout + proc.stderr
if token:
    output = output.replace(token, "***")
print(output)
del token, auth
assert proc.returncode == 0, "pip install failed; its output is above, with any token masked"

In [ ]:
# 2 - Provenance, read back from what is installed, never from the pin above
# (D-017, D-027). record() is every value an artifact carries; none of them is
# a URL or an environment variable, so none can carry the token (D-028).
import importlib

importlib.invalidate_caches()

from qecscreen.provenance import record

PROVENANCE = record()
for key, value in PROVENANCE.items():
    print(f"{key:16} {value}")
assert PROVENANCE["commit_sha"] == QECSCREEN_SHA, "the installed code is not the pinned commit (D-017)"

In [ ]:
# 3 - Configuration: constants only, nothing computed. Protocol values (p, shot
# budgets, decoder settings), the population and the wall budgets are the
# package's (qecscreen.protocol, qecscreen.evaluate.pilot). The runbook above
# says what to set each session.
RUN_NAME = "m0-pilot"
PREVIOUS = None  # session 1: None. Later: the attached input, e.g. "/kaggle/input/<this-notebook>"
COST_GATE = ""  # session 1: "". Later: the spec/evals.md §7 entry that approved the probe's cost report

In [ ]:
# 4 - Output directory: the pilot directory itself, cumulative across sessions.
# On Kaggle it is /kaggle/working/m0-pilot, which Save & Run All keeps as this
# version's output; the next session attaches that output and copies it back
# here. The artifact goes next to it, not inside it.
import datetime
import pathlib

CREATED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
KAGGLE_OUT = pathlib.Path("/kaggle/working")
OUT_DIR = (KAGGLE_OUT if KAGGLE_OUT.is_dir() else pathlib.Path("/content")) / RUN_NAME
OUT = OUT_DIR.parent / f"{RUN_NAME}-artifact-{CREATED_AT.replace(':', '')}.json"
print(OUT_DIR)

In [ ]:
# 5 - The one package call. Session 1 (no manifest in OUT_DIR) is the probe:
# 12 codes, 3 h. Later sessions run every unfinished code, longest first, for
# 10.5 h, and refuse to start without COST_GATE. Prints each worker's start
# and end, then the session summary; writes OUT_DIR/session_NNN.json.
from qecscreen.evaluate.pilot import PilotConfig, run_session

RESULT = run_session(PilotConfig(previous=PREVIOUS, cost_gate=COST_GATE or None), OUT_DIR)

In [ ]:
# 6 - Write the artifact and print a summary. Its provenance is record()'s dict,
# whole and unedited. The session's full summary, with its file inventory, is
# OUT_DIR/session_NNN.json.
import json

from qecscreen.evaluate.pilot import format_session

ARTIFACT = {
    "run_name": RUN_NAME,
    "created_at": CREATED_AT,
    "provenance": PROVENANCE,
    "result": {key: value for key, value in RESULT.items() if key != "inventory"},
}
OUT.write_text(json.dumps(ARTIFACT, indent=2, sort_keys=True), encoding="utf-8")
print(format_session(RESULT))
print("written:", OUT)